In [0]:
from datetime import date
from pyspark.sql.functions import *

In [0]:
from datetime import datetime
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/flight/schema')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/flight/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
display(df,
        checkpointLocation='/Volumes/projectdev/bronze/flight/checkpoint12/')

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("date", to_date(concat_ws("-",col('year'),lpad(col("Month"), 2, "0"),
        lpad(col("DayofMonth"), 2, "0")),'yyyy-MM-dd'))\
        .withColumn("DepTime",expr("gettimestamp(DepTime, 'HHmm')"))\
        .withColumn("CRSDepTime",expr("gettimestamp(CRSDepTime, 'HHmm')"))\
        .withColumn("ArrTime",expr("gettimestamp(ArrTime, 'HHmm')"))\
        .withColumn("CRSArrTime",expr("gettimestamp(CRSArrTime, 'HHmm')"))\
        .withColumn("ActualElapsedTime",col("ActualElapsedTime").cast('int'))\
        .withColumn("CRSElapsedTime",col("CRSElapsedTime").cast('int'))\
        .withColumn("AirTime",col("AirTime").cast('int'))\
        .withColumn("ArrDelay",col("ArrDelay").cast('int'))\
        .withColumn("DepDelay",col("DepDelay").cast('int'))\
        .withColumn("Origin",col("Origin"))\
        .withColumn("Dest",col("Dest"))\
        .withColumn("Distance",col("Distance").cast('double'))\
        .withColumn("TaxiIn",col("TaxiIn").cast('int'))\
        .withColumn("TaxiOut",col("TaxiOut").cast('int'))\
        .withColumn("Cancelled", expr("try_cast(Cancelled AS INT)"))\
        .withColumn("CancellationCode",col("CancellationCode"))\
        .withColumn("Diverted", expr("try_cast(Diverted AS INT)"))\
        .withColumn("CarrierDelay", expr("try_cast(CarrierDelay AS INT)"))\
        .withColumn("WeatherDelay",expr("try_cast(WeatherDelay AS INT)"))\
        .withColumn("NASDelay",expr("try_cast(NASDelay AS INT)"))\
        .withColumn("SecurityDelay",expr("try_cast(SecurityDelay AS INT)"))\
        .withColumn("LateAircraftDelay",expr("try_cast(LateAircraftDelay AS INT)"))\
        .drop('Year','Month','DayofMonth','DayOfWeek')\
        

# df.writeStream.trigger(once=True)\
#     .format('delta')\
#     .option("mergeSchema", "true")\
#     .outputMode('append')\
#     .option('checkpointLocation','/Volumes/projectdev/bronze/airport/checkpoint/')\
#     .start('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airport')
     

In [0]:
display(df)

In [0]:
%sql
create table if not exists projectdev.cleansed.airport
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airport'

In [0]:
%sql
select * from projectdev.cleansed.airport